# Capstone B: Numerical validation against symbolic truth

This starter notebook lets you explore polynomial, trigonometric, and Gaussian functions while comparing numerical derivatives and integrals to symbolic truth when SymPy is available.

## Setup

Import dependencies, define functions, and declare the optional SymPy block.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from math import erf
from calculus_with_python import differentiation, integration

try:
    import sympy as sp
    sympy_available = True
    sympy_x = sp.symbols("x")
except ImportError as exc:
    sympy_available = False
    sympy_error = exc
    sympy_x = None

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (10, 4)
np.set_printoptions(precision=4, suppress=True)


In [ ]:

def polynomial(x):
    return x**3 - 2 * x**2 + 0.5 * x - 1

def polynomial_derivative(x):
    return 3 * x**2 - 4 * x + 0.5

def polynomial_integral(a, b):
    return ((b**4 - a**4) / 4) - (2 / 3) * (b**3 - a**3) + 0.25 * (b**2 - a**2) - (b - a)

def trig(x):
    return np.sin(2 * x) + 0.5 * np.cos(3 * x)

def trig_derivative(x):
    return 2 * np.cos(2 * x) - 1.5 * np.sin(3 * x)

def trig_integral(a, b):
    return (-0.5) * (np.cos(2 * b) - np.cos(2 * a)) + (0.5 / 3) * (np.sin(3 * b) - np.sin(3 * a))

def gauss(x):
    return np.exp(-x**2)

def gauss_derivative(x):
    return -2 * x * np.exp(-x**2)

def gauss_integral(a, b):
    return np.sqrt(np.pi) / 2 * (erf(b) - erf(a))

function_specs = [
    {
        "name": "Cubic polynomial",
        "description": "f(x) = x^3 - 2x^2 + 0.5x - 1",
        "func": polynomial,
        "ref_derivative": polynomial_derivative,
        "ref_integral": polynomial_integral,
        "domain": (-1.5, 1.5),
        "sympy_expr": "x**3 - 2*x**2 + 0.5*x - 1",
    },
    {
        "name": "Trigonometric mix",
        "description": "f(x) = sin(2x) + 0.5cos(3x)",
        "func": trig,
        "ref_derivative": trig_derivative,
        "ref_integral": trig_integral,
        "domain": (-1.0, 2.0),
        "sympy_expr": "sin(2*x) + 0.5*cos(3*x)",
    },
    {
        "name": "Gaussian exp(-x^2)",
        "description": "f(x) = exp(-x^2)",
        "func": gauss,
        "ref_derivative": gauss_derivative,
        "ref_integral": gauss_integral,
        "domain": (-2.0, 2.0),
        "sympy_expr": "exp(-x**2)",
    },
]

for spec in function_specs:
    spec["x0"] = 0.5 * (spec["domain"][0] + spec["domain"][1])

if sympy_available:
    for spec in function_specs:
        sym_expr = sp.sympify(spec["sympy_expr"])
        spec["sympy_expr_obj"] = sym_expr
        spec["sympy_derivative_expr"] = sp.diff(sym_expr, sympy_x)
        spec["sympy_antiderivative"] = sp.integrate(sym_expr, sympy_x)
        spec["sympy_derivative_fn"] = sp.lambdify(sympy_x, spec["sympy_derivative_expr"], "numpy")
        antideriv_fn = sp.lambdify(sympy_x, spec["sympy_antiderivative"], "numpy")
        spec["sympy_integral_fn"] = lambda a, b, fn=antideriv_fn: float(fn(b) - fn(a))


## Visualizing the functions

Plot each function to build intuition before you start comparing errors.

In [ ]:

x_plot = np.linspace(-2.5, 2.5, 400)
plt.figure()
for spec in function_specs:
    plt.plot(x_plot, spec["func"](x_plot), label=spec["name"])
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Function gallery for symbolic comparison")
plt.legend()
plt.tight_layout()
plt.savefig("assets/figures/b_symbolic_truth/function_gallery.png")


## Symbolic expressions

Invoke SymPy expressions when present; otherwise the print statement explains why this section is skipped.

In [ ]:

if sympy_available:
    print("Sympy expressions are available for these functions:")
    for spec in function_specs:
        print(f"- {spec['name']}: derivative = {sp.pretty(spec['sympy_derivative_expr'])}; antiderivative = {sp.pretty(spec['sympy_antiderivative'])}")
else:
    print(
        "SymPy is not installed in this environment (", sympy_error, ").",
        "Symbolic expressions are skipped, but numeric comparisons rely on the curated reference formulas.",
    )


## Derivative error vs step size h

Plot forward, backward, and central difference schemes and note how their errors shrink (or plateau) as `h` decreases.

In [ ]:

h_methods = ["forward", "backward", "central"]
h_values = np.logspace(-4, -1, 20)
fig, axes = plt.subplots(1, len(function_specs), figsize=(15, 4), sharey=True)
if len(function_specs) == 1:
    axes = [axes]
for ax, spec in zip(axes, function_specs):
    x0 = spec["x0"]
    true_derivative = spec["ref_derivative"](x0)
    for method in h_methods:
        errors = []
        for h in h_values:
            estimate = differentiation.difference_quotient(spec["func"], x0, h=h, method=method)
            errors.append(abs(estimate - true_derivative))
        ax.loglog(h_values, errors, marker="o", label=method)
    ax.set_title(f"{spec['name']} at x≈{x0:.2f}")
    ax.set_xlabel("h")
    if ax is axes[0]:
        ax.set_ylabel("Absolute error")
    ax.grid(True, which="both", ls=":")
    if ax is axes[-1]:
        ax.legend(title="Method")
fig.suptitle("Derivative convergence for three finite-difference methods")
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("assets/figures/b_symbolic_truth/derivative_error.png")


## Integration error vs grid resolution n

Use trapezoid and Simpson rules to chart error decay as `n` increases. Comment on which rule is preferable for each function.

In [ ]:

n_values = [21, 41, 81, 161, 321]
integration_methods = ["trapezoid", "simpson"]
fig, axes = plt.subplots(1, len(function_specs), figsize=(15, 4), sharey=True)
if len(function_specs) == 1:
    axes = [axes]
for ax, spec in zip(axes, function_specs):
    a, b = spec["domain"]
    true_area = spec["ref_integral"](a, b)
    for method in integration_methods:
        errors = []
        for n in n_values:
            approx = integration.integrate_f(spec["func"], a, b, method=method, n=n)
            errors.append(abs(approx - true_area))
        ax.plot(n_values, errors, marker="o", label=method)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"Area vs n on [{a}, {b}] for {spec['name']}")
    ax.set_xlabel("Number of points (n)")
    if ax is axes[0]:
        ax.set_ylabel("Absolute error")
    ax.grid(True, which="both", ls=":")
    if ax is axes[-1]:
        ax.legend(title="Integration rule")
fig.suptitle("Integration error as grid resolution increases")
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("assets/figures/b_symbolic_truth/integration_error.png")


## Conclusions template

Summarize which derivative and integration methods you trust the most and how symbolic ground truth influences your confidence.